# Binary Hate / Non-Hate Video Transformer

This notebook trains a true video transformer classifier from extracted frame clips. It uses the same `video_df.csv` and `frames_unique` outputs as `analysis.ipynb`, but feeds stacked frame clips into a pretrained Hugging Face video model.

Default model: `MCG-NJU/videomae-base-finetuned-kinetics`.

Expected inputs:
- `E:/m-hvc/datasets/old-dataset/video_df.csv`
- `E:/m-hvc/datasets/old-dataset/frames_unique/{source_video_id}/*.jpg`

Outputs:
- `checkpoints/best_video_transformer.pt`
- `reports/video_transformer_metrics.csv`
- `reports/video_transformer_predictions.csv`
- `reports/video_transformer_history.csv`

In [12]:
from pathlib import Path
from copy import deepcopy
import math
import random
import warnings

import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from torchvision.transforms import functional as TF
from transformers import AutoConfig, AutoImageProcessor, VideoMAEForVideoClassification, get_cosine_schedule_with_warmup

warnings.filterwarnings("ignore", category=UserWarning)


In [13]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


PROJECT_ROOT = Path(r"E:/m-hvc")
DATASET_DIR = PROJECT_ROOT / "datasets" / "old-dataset"
MANIFEST_PATH = DATASET_DIR / "video_df.csv"
FRAMES_ROOT = DATASET_DIR / "frames_unique"

NOTEBOOK_DIR = PROJECT_ROOT / "multimodal-hvc" / "hate_non_hate_vision_classification"
CHECKPOINT_DIR = NOTEBOOK_DIR / "checkpoints_t"
REPORT_DIR = NOTEBOOK_DIR / "reports"
CHECKPOINT_PATH = CHECKPOINT_DIR / "best_video_transformer.pt"
LAST_CHECKPOINT_PATH = CHECKPOINT_DIR / "last_video_transformer.pt"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

CFG = {
    "seed": 42,
    "model_name": "MCG-NJU/videomae-base-finetuned-kinetics",
    "fallback_model_names": ["MCG-NJU/videomae-large-finetuned-kinetics"],
    "num_frames": 48,
    "image_size": 224,
    "batch_size": 4,
    "accum_steps": 12,
    "num_workers": 0,
    "epochs": 16,
    "freeze_epochs": 1,
    "unfreeze_last_n_blocks": 3,
    "val_size": 0.15,
    "test_size": 0.15,
    "head_lr": 1e-4,
    "backbone_lr": 3e-6,
    "weight_decay": 1e-2,
    "warmup_ratio": 0.08,
    "grad_clip": 0.8,
    "patience": 5,
    "ema_decay": 0.999,
    "use_weighted_sampler": True,
    "sampler_weight_cap": 4.0,
    "threshold_min": 0.10,
    "threshold_max": 0.90,
    "threshold_steps": 81,
    "label_smoothing": 0.04,
    "class_weight_power": 0.5,
    "clips_per_video_train": 1,
    "multi_clip_eval": 5,
    "temporal_dropout_prob": 0.15,
    "temporal_dropout_max_fraction": 0.25,
    "same_class_mix_prob": 0.08,
    "same_class_mix_alpha": 0.25,
    "random_erasing_prob": 0.15,
    "hard_example_finetune": False,
    "hard_example_boost": 2.0,
    "hard_example_epochs": 2,
    "use_gradient_checkpointing": True,
    "use_torch_compile": False,
}

set_seed(CFG["seed"])
print(f"device: {DEVICE}")
if torch.cuda.is_available():
    print(f"gpu: {torch.cuda.get_device_name(0)}")

device: cuda
gpu: NVIDIA GeForce RTX 4070


## Load Manifest And Frame Coverage

In [14]:
def numeric_frame_key(path: Path):
    stem = path.stem.split("_", 1)[0]
    return (0, int(stem)) if stem.isdigit() else (1, path.stem)


def list_frame_paths(frame_dir):
    frame_dir = Path(frame_dir)
    if not frame_dir.exists():
        return []
    paths = [path for path in frame_dir.iterdir() if path.suffix.lower() in {".jpg", ".jpeg", ".png"}]
    return sorted(paths, key=numeric_frame_key)


manifest_df = pd.read_csv(MANIFEST_PATH)
manifest_df["source_video_id"] = manifest_df["source_video_id"].astype(str)
manifest_df["label"] = manifest_df["label"].astype(int)
manifest_df["frame_dir"] = manifest_df.get(
    "frame_dir",
    manifest_df["source_video_id"].map(lambda value: str(FRAMES_ROOT / str(value))),
)
manifest_df["frame_dir"] = manifest_df.apply(
    lambda row: str(FRAMES_ROOT / str(row.source_video_id)) if pd.isna(row.frame_dir) else str(row.frame_dir),
    axis=1,
)
manifest_df["num_available_frames"] = manifest_df["frame_dir"].map(lambda value: len(list_frame_paths(value)))

missing_frames_df = manifest_df[manifest_df["num_available_frames"] <= 0].copy()
missing_frames_df.to_csv(REPORT_DIR / "video_transformer_missing_frame_dirs.csv", index=False)
clip_df = manifest_df[manifest_df["num_available_frames"] > 0].copy().reset_index(drop=True)

assert clip_df["source_video_id"].is_unique
assert set(clip_df["label"].unique()).issubset({0, 1})

print(f"trainable videos: {len(clip_df):,}")
print(f"missing frame dirs: {len(missing_frames_df):,}")
display(clip_df["label"].value_counts().rename_axis("label").reset_index(name="count"))
display(clip_df["num_available_frames"].describe().to_frame().T)


trainable videos: 3,281
missing frame dirs: 2


,label,count
0,0,1948
1,1,1333


,count,mean,std,min,25%,50%,75%,max
num_available_frames,3281.0,46.509296,23.358774,1.0,28.0,64.0,64.0,64.0


## Stratified Splits

In [15]:
train_val_df, test_df = train_test_split(
    clip_df,
    test_size=CFG["test_size"],
    random_state=CFG["seed"],
    stratify=clip_df["label"],
)
relative_val_size = CFG["val_size"] / (1.0 - CFG["test_size"])
train_df, val_df = train_test_split(
    train_val_df,
    test_size=relative_val_size,
    random_state=CFG["seed"],
    stratify=train_val_df["label"],
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

split_df = pd.concat(
    [train_df.assign(split="train"), val_df.assign(split="val"), test_df.assign(split="test")],
    ignore_index=True,
)
split_df.to_csv(REPORT_DIR / "video_transformer_splits.csv", index=False)

display(pd.concat([
    train_df["label"].value_counts().rename("train"),
    val_df["label"].value_counts().rename("val"),
    test_df["label"].value_counts().rename("test"),
], axis=1).fillna(0).astype(int))


,train,val,test
label,,,
0,1362,293,293
1,933,200,200


## Video Clip Dataset

In [16]:
def resolve_video_model_name(primary_name, fallback_names):
    candidates = [primary_name] + list(fallback_names)
    last_error = None
    for candidate in candidates:
        try:
            config = AutoConfig.from_pretrained(candidate)
            processor = AutoImageProcessor.from_pretrained(candidate)
            print(f"selected video transformer: {candidate}")
            return candidate, config, processor
        except Exception as exc:
            print(f"could not load {candidate}: {exc}")
            last_error = exc
    raise RuntimeError(f"Could not load any configured video transformer. Last error: {last_error}")


CFG["model_name"], model_config, image_processor = resolve_video_model_name(
    CFG["model_name"], CFG.get("fallback_model_names", [])
)

# VideoMAE positional embeddings are fixed to the pretrained clip length.
expected_frames = int(getattr(model_config, "num_frames", CFG["num_frames"]))
if CFG["num_frames"] != expected_frames:
    print(f"Overriding CFG['num_frames'] from {CFG['num_frames']} to model expected {expected_frames}")
    CFG["num_frames"] = expected_frames

processor_mean = tuple(float(x) for x in getattr(image_processor, "image_mean", [0.485, 0.456, 0.406]))
processor_std = tuple(float(x) for x in getattr(image_processor, "image_std", [0.229, 0.224, 0.225]))


def uniform_indices(num_items, num_frames):
    if num_items <= 1:
        return np.zeros(num_frames, dtype=np.int32)
    indices = np.linspace(0, num_items - 1, num=num_frames)
    return np.clip(np.round(indices).astype(np.int32), 0, num_items - 1)


def random_contiguous_indices(num_items, num_frames):
    if num_items <= num_frames:
        return uniform_indices(num_items, num_frames)
    start = np.random.randint(0, num_items - num_frames + 1)
    return np.arange(start, start + num_frames, dtype=np.int32)


def stride_jitter_indices(num_items, num_frames, stride=None):
    if num_items <= 1:
        return np.zeros(num_frames, dtype=np.int32)
    stride = int(stride or random.choice([1, 2, 3, 4]))
    span = (num_frames - 1) * stride + 1
    if num_items > span:
        start = np.random.randint(0, num_items - span + 1)
    else:
        start = np.random.randint(0, max(1, num_items))
    return np.clip(start + np.arange(num_frames) * stride, 0, num_items - 1).astype(np.int32)


def short_repeat_indices(num_items, num_frames):
    if num_items <= 1:
        return np.zeros(num_frames, dtype=np.int32)
    min_unique = max(2, int(round(num_frames * 0.25)))
    max_unique = max(min_unique, int(round(num_frames * 0.75)))
    unique_count = min(num_items, np.random.randint(min_unique, max_unique + 1))
    if num_items > unique_count:
        start = np.random.randint(0, num_items - unique_count + 1)
        unique = np.arange(start, start + unique_count, dtype=np.int32)
    else:
        unique = uniform_indices(num_items, unique_count)
    repeat_positions = np.linspace(0, len(unique) - 1, num=num_frames)
    return unique[np.clip(np.round(repeat_positions).astype(np.int32), 0, len(unique) - 1)]


def eval_view_indices(num_items, num_frames, view_index=0, num_views=1):
    if num_views <= 1 or num_items <= num_frames:
        return uniform_indices(num_items, num_frames)
    max_start = max(0, num_items - num_frames)
    start = int(round(max_start * view_index / max(1, num_views - 1)))
    return np.arange(start, start + num_frames, dtype=np.int32)


def sample_frame_paths(frame_dir, num_frames, train=False, strategy=None, view_index=0, num_views=1):
    frame_paths = list_frame_paths(frame_dir)
    if not frame_paths:
        raise FileNotFoundError(f"No frames found in {frame_dir}")

    if train:
        strategy = strategy or random.choice(["global_uniform", "random_contiguous", "stride_jitter", "short_repeat"])
        if strategy == "global_uniform":
            indices = uniform_indices(len(frame_paths), num_frames)
        elif strategy == "random_contiguous":
            indices = random_contiguous_indices(len(frame_paths), num_frames)
        elif strategy == "stride_jitter":
            indices = stride_jitter_indices(len(frame_paths), num_frames)
        elif strategy == "short_repeat":
            indices = short_repeat_indices(len(frame_paths), num_frames)
        else:
            raise ValueError(f"Unknown temporal sampling strategy: {strategy}")
    else:
        strategy = "multi_view_uniform" if num_views > 1 else "global_uniform"
        indices = eval_view_indices(len(frame_paths), num_frames, view_index=view_index, num_views=num_views)

    return [frame_paths[index] for index in indices], strategy


def temporal_dropout_clip(pixel_values, prob=0.0, max_fraction=0.25):
    if prob <= 0 or random.random() >= prob or pixel_values.shape[0] <= 1:
        return pixel_values
    clip = pixel_values.clone()
    num_frames = clip.shape[0]
    num_drop = max(1, int(round(num_frames * max_fraction * random.random())))
    drop_indices = np.random.choice(num_frames, size=min(num_drop, num_frames - 1), replace=False)
    for index in drop_indices:
        replacement = max(0, int(index) - 1) if index > 0 else min(num_frames - 1, int(index) + 1)
        clip[int(index)] = clip[replacement]
    return clip


class VideoTransform:
    def __init__(self, image_size=224, train=True):
        self.image_size = image_size
        self.train = train
        self.random_erasing = transforms.RandomErasing(
            p=float(CFG.get("random_erasing_prob", 0.0)),
            scale=(0.02, 0.12),
            ratio=(0.3, 3.3),
            value="random",
        )

    def __call__(self, images):
        processed = []
        if self.train:
            crop_i, crop_j, crop_h, crop_w = transforms.RandomResizedCrop.get_params(
                images[0], scale=(0.6, 1.0), ratio=(0.8, 1.25)
            )
            do_flip = random.random() < 0.5
            do_grayscale = random.random() < 0.08
            do_blur = random.random() < 0.10
            brightness = 1.0 + random.uniform(-0.25, 0.25)
            contrast = 1.0 + random.uniform(-0.25, 0.25)
            saturation = 1.0 + random.uniform(-0.20, 0.20)
            hue = random.uniform(-0.04, 0.04)
            blur_kernel = random.choice([3, 5])
        else:
            resize_size = int(self.image_size * 1.15)

        for image in images:
            image = image.convert("RGB")
            if self.train:
                image = TF.resized_crop(
                    image,
                    crop_i,
                    crop_j,
                    crop_h,
                    crop_w,
                    (self.image_size, self.image_size),
                    interpolation=InterpolationMode.BILINEAR,
                )
                if do_flip:
                    image = TF.hflip(image)
                image = TF.adjust_brightness(image, brightness)
                image = TF.adjust_contrast(image, contrast)
                image = TF.adjust_saturation(image, saturation)
                image = TF.adjust_hue(image, hue)
                if do_grayscale:
                    image = TF.rgb_to_grayscale(image, num_output_channels=3)
                if do_blur:
                    image = TF.gaussian_blur(image, kernel_size=[blur_kernel, blur_kernel], sigma=[0.1, 1.2])
            else:
                image = TF.resize(image, resize_size, interpolation=InterpolationMode.BILINEAR)
                image = TF.center_crop(image, [self.image_size, self.image_size])
            tensor = TF.to_tensor(image)
            tensor = TF.normalize(tensor, processor_mean, processor_std)
            if self.train:
                tensor = self.random_erasing(tensor)
            processed.append(tensor)
        return torch.stack(processed, dim=0)


class VideoTransformerDataset(Dataset):
    def __init__(self, frame_df, num_frames, image_size, train=False, clips_per_video=1):
        self.frame_df = frame_df.reset_index(drop=True)
        self.num_frames = int(num_frames)
        self.train = bool(train)
        self.clips_per_video = max(1, int(clips_per_video))
        self.transform = VideoTransform(image_size=image_size, train=train)

    def __len__(self):
        return len(self.frame_df)

    def _load_clip(self, frame_dir, view_index=0):
        frame_paths, strategy = sample_frame_paths(
            frame_dir,
            self.num_frames,
            train=self.train,
            view_index=view_index,
            num_views=self.clips_per_video,
        )
        images = []
        for frame_path in frame_paths:
            with Image.open(frame_path) as image:
                images.append(image.convert("RGB"))
        pixel_values = self.transform(images)
        if self.train:
            pixel_values = temporal_dropout_clip(
                pixel_values,
                prob=float(CFG.get("temporal_dropout_prob", 0.0)),
                max_fraction=float(CFG.get("temporal_dropout_max_fraction", 0.25)),
            )
        return pixel_values, strategy

    def __getitem__(self, index):
        row = self.frame_df.iloc[index]
        clips = []
        strategies = []
        for view_index in range(self.clips_per_video):
            pixel_values, strategy = self._load_clip(row.frame_dir, view_index=view_index)
            clips.append(pixel_values)
            strategies.append(strategy)
        pixel_values = clips[0] if self.clips_per_video == 1 else torch.stack(clips, dim=0)
        return {
            "pixel_values": pixel_values,
            "label": torch.tensor(int(row.label), dtype=torch.long),
            "source_video_id": str(row.source_video_id),
            "sampling_strategy": strategies[0] if len(strategies) == 1 else strategies,
        }

selected video transformer: MCG-NJU/videomae-base-finetuned-kinetics
Overriding CFG['num_frames'] from 48 to model expected 16


## DataLoaders

In [17]:
train_dataset = VideoTransformerDataset(
    train_df,
    CFG["num_frames"],
    CFG["image_size"],
    train=True,
    clips_per_video=CFG.get("clips_per_video_train", 1),
)
val_dataset = VideoTransformerDataset(
    val_df,
    CFG["num_frames"],
    CFG["image_size"],
    train=False,
    clips_per_video=CFG.get("multi_clip_eval", 1),
)
test_dataset = VideoTransformerDataset(
    test_df,
    CFG["num_frames"],
    CFG["image_size"],
    train=False,
    clips_per_video=CFG.get("multi_clip_eval", 1),
)

train_labels = train_df["label"].to_numpy(dtype=np.int64)
class_counts = np.bincount(train_labels, minlength=2).astype(np.float32)
class_weights = len(train_labels) / np.maximum(class_counts, 1.0)
loss_class_weights = class_weights ** float(CFG["class_weight_power"])
loss_class_weights = loss_class_weights / loss_class_weights.mean()
class_weight_tensor = torch.tensor(loss_class_weights, dtype=torch.float32, device=DEVICE)
sample_weights = np.asarray([class_weights[label] for label in train_labels], dtype=np.float32)
sample_weights = np.minimum(sample_weights, CFG["sampler_weight_cap"])

sampler = None
shuffle_train = True
if CFG["use_weighted_sampler"]:
    sampler = WeightedRandomSampler(torch.DoubleTensor(sample_weights), num_samples=len(sample_weights), replacement=True)
    shuffle_train = False


def video_collate_fn(samples):
    pixel_values = torch.stack([sample["pixel_values"] for sample in samples], dim=0)
    labels = torch.stack([sample["label"] for sample in samples], dim=0)
    ids = [sample["source_video_id"] for sample in samples]
    strategies = [sample["sampling_strategy"] for sample in samples]

    mix_prob = float(CFG.get("same_class_mix_prob", 0.0))
    if mix_prob > 0 and random.random() < mix_prob and pixel_values.ndim == 5:
        alpha = float(CFG.get("same_class_mix_alpha", 0.25))
        mixed = pixel_values.clone()
        for label in labels.unique().tolist():
            label_indices = (labels == label).nonzero(as_tuple=False).flatten()
            if len(label_indices) < 2:
                continue
            permuted = label_indices[torch.randperm(len(label_indices))]
            mixed[label_indices] = (1.0 - alpha) * mixed[label_indices] + alpha * pixel_values[permuted]
        pixel_values = mixed

    return {
        "pixel_values": pixel_values,
        "label": labels,
        "source_video_id": ids,
        "sampling_strategy": strategies,
    }


pin_memory = DEVICE.type == "cuda"
train_loader = DataLoader(
    train_dataset,
    batch_size=CFG["batch_size"],
    shuffle=shuffle_train,
    sampler=sampler,
    num_workers=CFG["num_workers"],
    pin_memory=pin_memory,
    drop_last=True,
    collate_fn=video_collate_fn,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=CFG["num_workers"],
    pin_memory=pin_memory,
    collate_fn=video_collate_fn,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=CFG["num_workers"],
    pin_memory=pin_memory,
    collate_fn=video_collate_fn,
)

sample_batch = next(iter(train_loader))
debug_strategies = [train_dataset[i]["sampling_strategy"] for i in range(min(32, len(train_dataset)))]
strategy_counts = pd.Series(debug_strategies).explode().value_counts().rename_axis("strategy").reset_index(name="count")
print(f"class_counts: non_hate={int(class_counts[0])}, hate={int(class_counts[1])}")
print(f"loss class weights: non_hate={class_weight_tensor[0].item():.4f}, hate={class_weight_tensor[1].item():.4f}")
print(f"train batches: {len(train_loader):,}, val batches: {len(val_loader):,}, test batches: {len(test_loader):,}")
print(f"sample pixel_values shape: {tuple(sample_batch['pixel_values'].shape)}")
print(f"model-required frames: {CFG['num_frames']}")
display(strategy_counts)
frame_axis = 2 if sample_batch["pixel_values"].ndim == 6 else 1
assert sample_batch["pixel_values"].shape[frame_axis] == CFG["num_frames"], "Frame count mismatch before model forward"

class_counts: non_hate=1362, hate=933
loss class weights: non_hate=0.9057, hate=1.0943
train batches: 573, val batches: 124, test batches: 124
sample pixel_values shape: (4, 16, 3, 224, 224)
model-required frames: 16


,strategy,count
0,stride_jitter,11
1,short_repeat,8
2,global_uniform,7
3,random_contiguous,6


## Model

In [18]:
id2label = {0: "non_hate", 1: "hate"}
label2id = {value: key for key, value in id2label.items()}

model_config.num_labels = 2
model_config.id2label = id2label
model_config.label2id = label2id

model = VideoMAEForVideoClassification.from_pretrained(
    CFG["model_name"],
    config=model_config,
    ignore_mismatched_sizes=True,
).to(DEVICE)

if CFG.get("use_gradient_checkpointing", False) and hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()
    print("gradient checkpointing: enabled")

if CFG.get("use_torch_compile", False) and hasattr(torch, "compile"):
    model = torch.compile(model)
    print("torch.compile: enabled")


def set_classifier_trainable(model, trainable=True):
    for name, parameter in model.named_parameters():
        if name.startswith("classifier"):
            parameter.requires_grad_(trainable)


def set_backbone_frozen(model):
    for name, parameter in model.named_parameters():
        if not name.startswith("classifier"):
            parameter.requires_grad_(False)
    set_classifier_trainable(model, True)


def unfreeze_last_encoder_blocks(model, last_n_blocks=3):
    set_backbone_frozen(model)
    blocks = getattr(getattr(model, "videomae", None), "encoder", None)
    blocks = getattr(blocks, "layer", None)
    if blocks is None:
        raise AttributeError("Could not locate VideoMAE encoder blocks at model.videomae.encoder.layer")
    last_n_blocks = max(0, min(int(last_n_blocks), len(blocks)))
    for block in blocks[-last_n_blocks:]:
        for parameter in block.parameters():
            parameter.requires_grad_(True)
    return last_n_blocks


def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


set_backbone_frozen(model)
if CFG["freeze_epochs"] <= 0:
    unfrozen_blocks = unfreeze_last_encoder_blocks(model, CFG.get("unfreeze_last_n_blocks", 3))
    print(f"initially unfroze last {unfrozen_blocks} VideoMAE encoder blocks")
else:
    print("initially frozen VideoMAE backbone; classifier/head remains trainable")
print(f"model: {CFG['model_name']}")
print(f"trainable parameters: {count_trainable_parameters(model):,}")
print(f"total parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 162/162 [00:00<00:00, 5761.85it/s]
[transformers] VideoMAEForVideoClassification LOAD REPORT from: MCG-NJU/videomae-base-finetuned-kinetics
Key                                                            | Status     | Details                                                                                 
---------------------------------------------------------------+------------+-----------------------------------------------------------------------------------------
videomae.encoder.layer.{0...11}.attention.attention.v_bias     | UNEXPECTED |                                                                                         
videomae.encoder.layer.{0...11}.attention.attention.q_bias     | UNEXPECTED |

gradient checkpointing: enabled
initially frozen VideoMAE backbone; classifier/head remains trainable
model: MCG-NJU/videomae-base-finetuned-kinetics
trainable parameters: 1,538
total parameters: 86,237,954


## Optimizer, Scheduler, EMA

In [19]:
def split_decay(named_params):
    decay = []
    no_decay = []
    for name, parameter in named_params:
        if parameter.ndim <= 1 or name.endswith(".bias") or "norm" in name.lower():
            no_decay.append(parameter)
        else:
            decay.append(parameter)
    return decay, no_decay


def build_optimizer(model):
    backbone_named = []
    head_named = []
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        if name.startswith("classifier"):
            head_named.append((name, parameter))
        else:
            backbone_named.append((name, parameter))

    param_groups = []
    for named_params, lr in ((backbone_named, CFG["backbone_lr"]), (head_named, CFG["head_lr"])):
        decay, no_decay = split_decay(named_params)
        if decay:
            param_groups.append({"params": decay, "lr": lr, "weight_decay": CFG["weight_decay"]})
        if no_decay:
            param_groups.append({"params": no_decay, "lr": lr, "weight_decay": 0.0})
    return torch.optim.AdamW(param_groups, betas=(0.9, 0.999))


def create_ema_model(model):
    ema_model = deepcopy(model).eval()
    for parameter in ema_model.parameters():
        parameter.requires_grad_(False)
    return ema_model


def update_ema(model, ema_model, decay):
    with torch.no_grad():
        model_state = model.state_dict()
        ema_state = ema_model.state_dict()
        for key, ema_value in ema_state.items():
            model_value = model_state[key].detach()
            if torch.is_floating_point(ema_value):
                ema_value.mul_(decay).add_(model_value, alpha=1.0 - decay)
            else:
                ema_value.copy_(model_value)


steps_per_epoch = max(1, math.ceil(len(train_loader) / max(1, CFG["accum_steps"])))
optimizer = build_optimizer(model)
initial_steps = steps_per_epoch * max(1, CFG["freeze_epochs"])
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, int(initial_steps * CFG["warmup_ratio"])),
    num_training_steps=max(1, initial_steps),
)
scaler = GradScaler(enabled=USE_AMP)
ema_model = create_ema_model(model)


## Metrics And Training Helpers

In [20]:
def safe_auc(metric_fn, y_true, y_prob):
    try:
        if len(np.unique(y_true)) < 2:
            return np.nan
        return float(metric_fn(y_true, y_prob))
    except Exception:
        return np.nan


def compute_metrics(y_true, y_prob, threshold=0.5, prefix=""):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= float(threshold)).astype(int)
    return {
        f"{prefix}threshold": float(threshold),
        f"{prefix}accuracy": float(accuracy_score(y_true, y_pred)),
        f"{prefix}f1": float(f1_score(y_true, y_pred, zero_division=0)),
        f"{prefix}precision": float(precision_score(y_true, y_pred, zero_division=0)),
        f"{prefix}recall": float(recall_score(y_true, y_pred, zero_division=0)),
        f"{prefix}roc_auc": safe_auc(roc_auc_score, y_true, y_prob),
        f"{prefix}pr_auc": safe_auc(average_precision_score, y_true, y_prob),
    }


def optimize_threshold(y_true, y_prob):
    thresholds = np.linspace(CFG["threshold_min"], CFG["threshold_max"], CFG["threshold_steps"])
    best_threshold = 0.5
    best_f1 = -1.0
    for threshold in thresholds:
        score = f1_score(y_true, (y_prob >= threshold).astype(int), zero_division=0)
        if score > best_f1:
            best_f1 = float(score)
            best_threshold = float(threshold)
    return best_threshold, best_f1


def forward_logits(model, pixel_values):
    if pixel_values.ndim == 6:
        batch_size, num_clips, num_frames, channels, height, width = pixel_values.shape
        flat_pixel_values = pixel_values.view(batch_size * num_clips, num_frames, channels, height, width)
        logits = model(pixel_values=flat_pixel_values).logits
        return logits.view(batch_size, num_clips, -1)
    return model(pixel_values=pixel_values).logits


def run_epoch(model, loader, optimizer=None, scheduler=None, scaler=None, ema_model=None):
    train_mode = optimizer is not None
    model.train(train_mode)
    total_loss = 0.0
    total_examples = 0
    all_labels = []
    all_probs = []
    all_ids = []

    if train_mode:
        optimizer.zero_grad(set_to_none=True)

    progress = tqdm(loader, leave=False, desc="train" if train_mode else "eval")
    for step, batch in enumerate(progress, start=1):
        pixel_values = batch["pixel_values"].to(DEVICE, non_blocking=True)
        labels = batch["label"].to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(train_mode):
            with autocast(device_type=DEVICE.type, enabled=USE_AMP):
                logits = forward_logits(model, pixel_values)
                if logits.ndim == 3:
                    loss_logits = logits.reshape(-1, logits.shape[-1])
                    loss_labels = labels.repeat_interleave(logits.shape[1])
                    prob_values = torch.softmax(logits, dim=-1).mean(dim=1)[:, 1]
                else:
                    loss_logits = logits
                    loss_labels = labels
                    prob_values = torch.softmax(logits, dim=1)[:, 1]
                loss = F.cross_entropy(
                    loss_logits,
                    loss_labels,
                    weight=class_weight_tensor,
                    label_smoothing=float(CFG.get("label_smoothing", 0.0)),
                )
                loss_for_backward = loss / max(1, CFG["accum_steps"])

            if train_mode:
                scaler.scale(loss_for_backward).backward()
                should_step = step % CFG["accum_steps"] == 0 or step == len(loader)
                if should_step:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
                    if scheduler is not None:
                        scheduler.step()
                    if ema_model is not None:
                        update_ema(model, ema_model, CFG["ema_decay"])

        probs = prob_values.detach().cpu().numpy()
        batch_size = labels.shape[0]
        total_loss += float(loss.detach().cpu().item()) * batch_size
        total_examples += batch_size
        all_probs.append(probs)
        all_labels.append(labels.detach().cpu().numpy())
        all_ids.extend(batch["source_video_id"])
        progress.set_postfix(loss=total_loss / max(1, total_examples))

    y_true = np.concatenate(all_labels).astype(int)
    y_prob = np.concatenate(all_probs).astype(float)
    assert len(all_ids) == len(y_prob), "Expected exactly one probability per source video"
    return total_loss / max(1, total_examples), y_true, y_prob, all_ids


def save_checkpoint(path, model, threshold, metrics, epoch):
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "config": CFG,
            "threshold": float(threshold),
            "metrics": metrics,
            "epoch": int(epoch),
            "model_name": CFG["model_name"],
            "id2label": id2label,
            "label2id": label2id,
        },
        path,
    )


## Baselines

In [21]:
val_prior = np.full(len(val_df), train_df["label"].mean(), dtype=np.float32)
majority_prob = np.full(len(val_df), 0.0 if train_df["label"].mean() < 0.5 else 1.0, dtype=np.float32)
baseline_df = pd.DataFrame(
    [
        {"baseline": "majority_class", **compute_metrics(val_df["label"].values, majority_prob, 0.5)},
        {"baseline": "train_prior_prob", **compute_metrics(val_df["label"].values, val_prior, 0.5)},
    ]
)
baseline_df.to_csv(REPORT_DIR / "video_transformer_baselines.csv", index=False)
display(baseline_df)


,baseline,threshold,accuracy,f1,precision,recall,roc_auc,pr_auc
0,majority_class,0.5,0.59432,0.0,0.0,0.0,0.5,0.40568
1,train_prior_prob,0.5,0.59432,0.0,0.0,0.0,0.5,0.40568


## Train

In [22]:
best_score = -1.0
history = []
epochs_without_improvement = 0
best_val_payload = None

for epoch in range(1, CFG["epochs"] + 1):
    if CFG["freeze_epochs"] > 0 and epoch == CFG["freeze_epochs"] + 1:
        unfrozen_blocks = unfreeze_last_encoder_blocks(model, CFG.get("unfreeze_last_n_blocks", 3))
        optimizer = build_optimizer(model)
        remaining_steps = steps_per_epoch * max(1, CFG["epochs"] - epoch + 1)
        scheduler = get_cosine_schedule_with_warmup(
            optimizer,
            num_warmup_steps=max(1, int(remaining_steps * CFG["warmup_ratio"])),
            num_training_steps=max(1, remaining_steps),
        )
        print(f"epoch {epoch:02d}: unfroze last {unfrozen_blocks} VideoMAE encoder blocks")

    train_loss, train_y, train_prob, _ = run_epoch(
        model,
        train_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        ema_model=ema_model,
    )
    eval_model = ema_model if ema_model is not None else model
    val_loss, val_y, val_prob, val_ids = run_epoch(eval_model, val_loader)

    train_metrics = compute_metrics(train_y, train_prob, threshold=0.5, prefix="train_")
    val_metrics = compute_metrics(val_y, val_prob, threshold=0.5, prefix="val_")

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        **train_metrics,
        **val_metrics,
    }
    history.append(row)
    pd.DataFrame(history).to_csv(REPORT_DIR / "video_transformer_history.csv", index=False)

    print(
        f"epoch {epoch:02d} | train_loss={train_loss:.4f} | train_acc={train_metrics['train_accuracy']:.4f} | "
        f"train_f1={train_metrics['train_f1']:.4f} | val_loss={val_loss:.4f} | "
        f"val_acc@0.5={val_metrics['val_accuracy']:.4f} | val_f1@0.5={val_metrics['val_f1']:.4f}"
    )

    score = val_metrics["val_f1"]
    if score > best_score:
        best_score = score
        epochs_without_improvement = 0
        best_val_payload = (val_loss, val_y.copy(), val_prob.copy(), list(val_ids))
        save_checkpoint(CHECKPOINT_PATH, eval_model, 0.5, val_metrics, epoch)
        print(f"saved best checkpoint: {CHECKPOINT_PATH}")
    else:
        epochs_without_improvement += 1

    save_checkpoint(LAST_CHECKPOINT_PATH, eval_model, 0.5, val_metrics, epoch)

    if epochs_without_improvement >= CFG["patience"]:
        print(f"early stopping after {CFG['patience']} epochs without improvement")
        break

history_df = pd.DataFrame(history)
display(history_df.tail())

KeyboardInterrupt: 

## Test Evaluation

In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)
model.eval()

val_loss, val_y, val_prob, val_ids = run_epoch(model, val_loader)
best_threshold, best_val_f1 = optimize_threshold(val_y, val_prob)
val_default_metrics = compute_metrics(val_y, val_prob, threshold=0.5, prefix="val_default_")
val_tuned_metrics = compute_metrics(val_y, val_prob, threshold=best_threshold, prefix="val_tuned_")
save_checkpoint(CHECKPOINT_PATH, model, best_threshold, val_tuned_metrics, int(checkpoint.get("epoch", -1)))

val_predictions_df = pd.DataFrame(
    {
        "source_video_id": val_ids,
        "label": val_y.astype(int),
        "prob_hate": val_prob,
        "pred_default": (val_prob >= 0.5).astype(int),
        "pred_tuned": (val_prob >= best_threshold).astype(int),
    }
)
val_predictions_df.to_csv(REPORT_DIR / "video_transformer_val_predictions.csv", index=False)
pd.DataFrame([{"threshold": best_threshold, "val_f1": best_val_f1}]).to_csv(
    REPORT_DIR / "video_transformer_final_threshold.csv",
    index=False,
)

hard_val_df = val_predictions_df[val_predictions_df["label"] != val_predictions_df["pred_tuned"]].copy()
hard_val_df["error_type"] = np.where(hard_val_df["label"] == 1, "false_negative", "false_positive")
hard_val_df.to_csv(REPORT_DIR / "video_transformer_hard_examples_val.csv", index=False)

if CFG.get("hard_example_finetune", False) and len(hard_val_df) > 0:
    print("hard-example fine-tuning is enabled; boost train/validation mistakes in a short second stage")
    train_loss_eval, train_y_eval, train_prob_eval, train_ids_eval = run_epoch(model, train_loader)
    train_pred_eval = (train_prob_eval >= best_threshold).astype(int)
    hard_train_ids = {vid for vid, y, pred in zip(train_ids_eval, train_y_eval, train_pred_eval) if int(y) != int(pred)}
    boosted_weights = sample_weights.copy()
    train_id_array = train_df["source_video_id"].astype(str).to_numpy()
    boosted_weights[np.isin(train_id_array, list(hard_train_ids))] *= float(CFG.get("hard_example_boost", 2.0))
    hard_sampler = WeightedRandomSampler(torch.DoubleTensor(boosted_weights), num_samples=len(boosted_weights), replacement=True)
    hard_loader = DataLoader(
        train_dataset,
        batch_size=CFG["batch_size"],
        sampler=hard_sampler,
        shuffle=False,
        num_workers=CFG["num_workers"],
        pin_memory=pin_memory,
        drop_last=True,
        collate_fn=video_collate_fn,
    )
    optimizer = build_optimizer(model)
    hard_steps = max(1, math.ceil(len(hard_loader) / max(1, CFG["accum_steps"]))) * int(CFG.get("hard_example_epochs", 2))
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=max(1, int(hard_steps * CFG["warmup_ratio"])),
        num_training_steps=hard_steps,
    )
    for hard_epoch in range(1, int(CFG.get("hard_example_epochs", 2)) + 1):
        hard_loss, _, _, _ = run_epoch(model, hard_loader, optimizer=optimizer, scheduler=scheduler, scaler=scaler, ema_model=ema_model)
        print(f"hard-example epoch {hard_epoch:02d} | train_loss={hard_loss:.4f}")
    model.load_state_dict((ema_model if ema_model is not None else model).state_dict())
    val_loss, val_y, val_prob, val_ids = run_epoch(model, val_loader)
    best_threshold, best_val_f1 = optimize_threshold(val_y, val_prob)
    val_tuned_metrics = compute_metrics(val_y, val_prob, threshold=best_threshold, prefix="val_tuned_")
    save_checkpoint(CHECKPOINT_PATH, model, best_threshold, val_tuned_metrics, int(checkpoint.get("epoch", -1)))


test_loss, test_y, test_prob, test_ids = run_epoch(model, test_loader)
assert len(test_prob) == len(test_ids) == len(test_df), "Multi-view test inference should produce one probability per video"
test_default_metrics = compute_metrics(test_y, test_prob, threshold=0.5, prefix="test_default_")
test_tuned_metrics = compute_metrics(test_y, test_prob, threshold=best_threshold, prefix="test_tuned_")

metrics_df = pd.DataFrame(
    [
        {"split": "val", "threshold_type": "default", "loss": val_loss, **val_default_metrics},
        {"split": "val", "threshold_type": "final_val_tuned", "loss": val_loss, **val_tuned_metrics},
        {"split": "test", "threshold_type": "default", "loss": test_loss, **test_default_metrics},
        {"split": "test", "threshold_type": "final_val_tuned", "loss": test_loss, **test_tuned_metrics},
    ]
)
metrics_df.to_csv(REPORT_DIR / "video_transformer_metrics.csv", index=False)

predictions_df = pd.DataFrame(
    {
        "source_video_id": test_ids,
        "label": test_y.astype(int),
        "prob_hate": test_prob,
        "pred_default": (test_prob >= 0.5).astype(int),
        "pred_tuned": (test_prob >= best_threshold).astype(int),
    }
)
predictions_df.to_csv(REPORT_DIR / "video_transformer_predictions.csv", index=False)

hard_test_df = predictions_df[predictions_df["label"] != predictions_df["pred_tuned"]].copy()
hard_test_df["error_type"] = np.where(hard_test_df["label"] == 1, "false_negative", "false_positive")
hard_test_df.to_csv(REPORT_DIR / "video_transformer_hard_examples_test.csv", index=False)

print(f"final validation threshold selected once after training: {best_threshold:.3f}")
display(metrics_df)
display(predictions_df.head())

## Plots

In [ ]:
history_df = pd.read_csv(REPORT_DIR / "video_transformer_history.csv")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
axes[0].plot(history_df["epoch"], history_df["val_loss"], label="val_loss")
axes[0].set_title("Loss")
axes[0].set_xlabel("epoch")
axes[0].legend()

axes[1].plot(history_df["epoch"], history_df["train_accuracy"], label="train_accuracy")
axes[1].plot(history_df["epoch"], history_df["train_f1"], label="train_f1")
axes[1].plot(history_df["epoch"], history_df["val_accuracy"], label="val_accuracy@0.5")
axes[1].plot(history_df["epoch"], history_df["val_f1"], label="val_f1@0.5")
axes[1].set_title("Metrics")
axes[1].set_xlabel("epoch")
axes[1].legend()
fig.tight_layout()
fig.savefig(REPORT_DIR / "video_transformer_training_curves.png", dpi=160)
plt.show()

cm = confusion_matrix(test_y, (test_prob >= best_threshold).astype(int), labels=[0, 1])
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1], labels=["non_hate", "hate"])
ax.set_yticks([0, 1], labels=["non_hate", "hate"])
ax.set_xlabel("predicted")
ax.set_ylabel("actual")
ax.set_title(f"Video Transformer Confusion Matrix @ threshold={best_threshold:.3f}")
for row in range(cm.shape[0]):
    for col in range(cm.shape[1]):
        ax.text(col, row, str(cm[row, col]), ha="center", va="center", color="black")
fig.colorbar(im, ax=ax)
fig.tight_layout()
fig.savefig(REPORT_DIR / "video_transformer_confusion_matrix.png", dpi=160)
plt.show()

## Inference Utility

In [ ]:
def load_video_transformer(checkpoint_path=CHECKPOINT_PATH):
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    model_name = checkpoint.get("model_name", CFG["model_name"])
    inference_config = AutoConfig.from_pretrained(model_name)
    inference_config.num_labels = 2
    inference_config.id2label = checkpoint.get("id2label", id2label)
    inference_config.label2id = checkpoint.get("label2id", label2id)
    inference_model = VideoMAEForVideoClassification.from_pretrained(
        model_name,
        config=inference_config,
        ignore_mismatched_sizes=True,
    ).to(DEVICE)
    inference_model.load_state_dict(checkpoint["model_state_dict"])
    inference_model.eval()
    return inference_model, float(checkpoint.get("threshold", 0.5)), checkpoint.get("config", CFG)


def predict_frame_dir(frame_dir, checkpoint_path=CHECKPOINT_PATH, num_views=None):
    inference_model, threshold, inference_cfg = load_video_transformer(checkpoint_path)
    num_views = int(num_views or inference_cfg.get("multi_clip_eval", 1))
    clips = []
    for view_index in range(max(1, num_views)):
        frame_paths, _ = sample_frame_paths(
            frame_dir,
            inference_cfg["num_frames"],
            train=False,
            view_index=view_index,
            num_views=max(1, num_views),
        )
        images = []
        for frame_path in frame_paths:
            with Image.open(frame_path) as image:
                images.append(image.convert("RGB"))
        clips.append(VideoTransform(inference_cfg["image_size"], train=False)(images))
    pixel_values = torch.stack(clips, dim=0).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        with autocast(device_type=DEVICE.type, enabled=USE_AMP):
            logits = forward_logits(inference_model, pixel_values)
            prob = torch.softmax(logits, dim=-1).mean(dim=1)[:, 1].item()
    return {"prob_hate": float(prob), "threshold": float(threshold), "prediction": int(prob >= threshold)}
